<a href="https://colab.research.google.com/github/wiz124/FN14_Tweak/blob/master/openmmtest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
#!pip install -q openmm
#!pip install -q openmm[cuda12]
#!pip install -q openmm[hip6]
#!pip install -1 pdbfixer

from openmm.app import *
from openmm import *
from openmm.unit import *
from sys import stdout
from pdbfixer import PDBFixer


fixer = PDBFixer(filename='input.pdb')
fixer.findMissingResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(7.0)

with open('fixed.pdb', 'w') as f:
    PDBFile.writeFile(fixer.topology, fixer.positions, f)

pdb = PDBFile('fixed.pdb')
modeller=Modeller(pdb.topology, pdb.positions)
forcefield = ForceField('amber19-all.xml', 'amber19/tip3pfb.xml')

modeller.addHydrogens(forcefield)
modeller.addSolvent(forcefield)
# modeller.addMembrane(forcefield, lipidType='POPC', minimumPadding=1*nanometer) #option for membrane proteins, fancy....
modeller.deleteWater()


system = forcefield.createSystem(pdb.topology, nonbondedMethod=PME,
        nonbondedCutoff=1*nanometer, constraints=HBonds)

integrator = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, 0.004*picoseconds)
simulation = Simulation(pdb.topology, system, integrator)
simulation.context.setPositions(pdb.positions)
simulation.minimizeEnergy()
simulation.reporters.append(DCDReporter('output.dcd', 1000))
simulation.reporters.append(StateDataReporter(stdout, 1000, step=True,
        potentialEnergy=True, temperature=True))
simulation.step(10000)


AttributeError: 'NoneType' object has no attribute 'value_in_unit'